# Chapter 4 &mdash; DFA Everywhere

**Concept 1 of the Chapter 4 decomposition:** *DFA Everywhere: Why Finite-State Machines Matter*

Lexers, traffic lights, pilot mode confusion, deep packet inspection &mdash; finite-state machines are already everywhere.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-DFA-Everywhere/Concept-DFA-Everywhere.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Finite-state machines run **lexers** in compilers, **traffic lights**, **airplane
control panels** (and the pilot's own mental model &mdash; a mismatch is called *mode
confusion*), and **deep packet inspection** hardware.

Speed matters there for a security reason: a front-end machine too slow to keep up
invites a **denial-of-service** attack.

Variants you will meet elsewhere &mdash; Mealy machines, communicating FSMs, UML
statecharts, B&uuml;chi automata &mdash; are all variations on what we study here.

## 2. Definitions

### A traffic light as a DFA

Four states, cycling on a `t` (timer) event, with a `p` (pedestrian button).

In [ ]:
light = md2mc('''DFA
IF   : t -> Grn
IF   : p -> IF
Grn  : t -> Yel
Grn  : p -> Grn
Yel  : t -> IF
Yel  : p -> Yel
''')
print("states :", sorted(light["Q"]))
assert light["q0"] == "IF"

### A malware-signature scanner

One pass, constant memory, never backs up.

In [ ]:
dpi = md2mc('''DFA
I    : 1 -> S1
I    : 0 -> I
S1   : 1 -> S11
S1   : 0 -> I
S11  : 1 -> S11
S11  : 0 -> S110
S110 : 1 -> F
S110 : 0 -> I
F    : 0 | 1 -> F
''')
print("signature scanner states :", sorted(dpi["Q"]))

## 3. Tests

The light cycles forever and never grows.

In [ ]:
seq = 'ttt' * 3
print("after", seq, "-> accepted?", accepts_dfa(light, seq))
assert accepts_dfa(light, 'ttt')    # back to the start state
print("states used: %d, however long the input" % len(light["Q"]))

The scanner finds the signature in one pass.

In [ ]:
for pkt in ['0001101000', '1111', '1101', '0011010']:
    print("%-12s contains 1101? %s" % (pkt, accepts_dfa(dpi, pkt)))
assert accepts_dfa(dpi, '1101') and not accepts_dfa(dpi, '1111')

## 4. Animation

Watch the scanner fall back correctly on a mismatch &mdash; that is what lets it make one pass.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(dpi, FuseEdges=True)

## 5. Exercises


1. Add an `emergency` input to the traffic light that jumps straight to red.
2. Change the signature to `1011`. Which fall-back edges change?
3. Why would a *slow* scanner be a security problem and not just an annoyance?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter4/Concept-DFA-Everywhere')